In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [2]:
# Cell 1: Bronze jadvaldan sensor_id larni olish
from pyspark.sql import SparkSession
import requests
import json
from datetime import datetime

bronze_df = spark.sql("SELECT DISTINCT sensor_id FROM dbo.bronze_openaq_locations WHERE sensor_id IS NOT NULL")
sensor_ids = [row.sensor_id for row in bronze_df.collect()]

print(f"Topilgan sensor_id lar soni: {len(sensor_ids)}")
print(type(sensor_ids[0]))

StatementMeta(, 67340c2a-d742-46cd-aaea-d1a4c60f5af5, 4, Finished, Available, Finished, False)

Topilgan sensor_id lar soni: 188
<class 'int'>


In [4]:
# Cell 2: Rate limit ni kuzatib, 2025 yilgi ma'lumotlarni olish
import requests
import json
import time

all_measurements = []

# Oyma-oy so'rash — 408 timeout xatosini oldini olish
months = [
    ("2025-01-01T00:00:00Z", "2025-01-31T23:59:59Z"),
    ("2025-02-01T00:00:00Z", "2025-02-28T23:59:59Z"),
    ("2025-03-01T00:00:00Z", "2025-03-31T23:59:59Z"),
    ("2025-04-01T00:00:00Z", "2025-04-30T23:59:59Z"),
    ("2025-05-01T00:00:00Z", "2025-05-31T23:59:59Z"),
    ("2025-06-01T00:00:00Z", "2025-06-30T23:59:59Z"),
    ("2025-07-01T00:00:00Z", "2025-07-31T23:59:59Z"),
    ("2025-08-01T00:00:00Z", "2025-08-31T23:59:59Z"),
    ("2025-09-01T00:00:00Z", "2025-09-30T23:59:59Z"),
    ("2025-10-01T00:00:00Z", "2025-10-31T23:59:59Z"),
    ("2025-11-01T00:00:00Z", "2025-11-30T23:59:59Z"),
    ("2025-12-01T00:00:00Z", "2025-12-31T23:59:59Z"),
]

request_headers = {"X-API-Key": "c2624cb4749dce035d38bfb55ba36c921ada8a72aa856f6b69c16f0bd0574181"}

total   = len(sensor_ids) * len(months)
current = 0

for i, sensor_id in enumerate(sensor_ids):
    for date_from, date_to in months:
        current += 1
        url = f"https://api.openaq.org/v3/sensors/{sensor_id}/days"

        params = {
            "date_from": date_from,
            "date_to":   date_to,
            "limit":     100
        }

        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, headers=request_headers, timeout=30)

                # Rate limit headerlarini o'qish
                headers      = response.headers
                rl_limit     = int(headers.get("x-ratelimit-limit",     headers.get("ratelimit-limit",     60)))
                rl_remaining = int(headers.get("x-ratelimit-remaining", headers.get("ratelimit-remaining",  60)))
                rl_reset     = int(headers.get("x-ratelimit-reset",     headers.get("ratelimit-reset",      1)))

                print(f"[{current}/{total}] Sensor {sensor_id} | {date_from[:7]} | "
                      f"Remaining: {rl_remaining} | Reset: {rl_reset}s")

                # 429 — rate limit
                if response.status_code == 429:
                    wait = rl_reset if rl_reset > 0 else 60
                    print(f"🚦 Rate limit! {wait}s kutilmoqda...")
                    time.sleep(wait)
                    continue  # qayta urinish

                # 408 — timeout
                elif response.status_code == 408:
                    print(f"⏱️ 408 Timeout! Qayta urinish ({attempt+1}/{max_retries})...")
                    time.sleep(5)
                    continue

                elif response.status_code == 200:
                    data    = response.json()
                    results = data.get("results", [])
                    for record in results:
                        record["sensor_id"] = sensor_id
                        all_measurements.append(record)
                    print(f"   ✅ {len(results)} ta yozuv olindi")

                    # Rate limit ga qarab sleep — break DAN OLDIN
                    if rl_remaining <= 5:
                        wait = rl_reset if rl_reset > 0 else 60
                        print(f"⏳ Remaining={rl_remaining} — {wait}s kutilmoqda...")
                        time.sleep(wait)
                    elif rl_remaining <= 20:
                        time.sleep(2)
                    else:
                        time.sleep(0.5)

                    break  # muvaffaqiyatli — keyingi oyga o'tish

                else:
                    print(f"   ❌ Xatolik: {response.status_code} - {response.text[:100]}")
                    break

            except requests.exceptions.Timeout:
                print(f"⏰ Sensor {sensor_id} {date_from[:7]}: Timeout! Qayta urinish ({attempt+1}/{max_retries})...")
                time.sleep(10)
                continue
            except requests.exceptions.ConnectionError as e:
                print(f"🌐 Sensor {sensor_id}: Ulanish xatosi - {str(e)[:100]}")
                time.sleep(5)
                continue
            except Exception as e:
                print(f"⚠️ Sensor {sensor_id}: {str(e)}")
                break

print(f"\n📊 Jami yig'ilgan yozuvlar: {len(all_measurements)}")

StatementMeta(, 67340c2a-d742-46cd-aaea-d1a4c60f5af5, 6, Submitted, Running, Running, True)

[1/2256] Sensor 9648796 | 2025-01 | Remaining: 59 | Reset: 60s
   ✅ 12 ta yozuv olindi
[2/2256] Sensor 9648796 | 2025-02 | Remaining: 58 | Reset: 58s
   ✅ 18 ta yozuv olindi
[3/2256] Sensor 9648796 | 2025-03 | Remaining: 57 | Reset: 57s
   ✅ 31 ta yozuv olindi
[4/2256] Sensor 9648796 | 2025-04 | Remaining: 56 | Reset: 55s
   ✅ 27 ta yozuv olindi
[5/2256] Sensor 9648796 | 2025-05 | Remaining: 55 | Reset: 54s
   ✅ 31 ta yozuv olindi
[6/2256] Sensor 9648796 | 2025-06 | Remaining: 54 | Reset: 53s
   ✅ 30 ta yozuv olindi
[7/2256] Sensor 9648796 | 2025-07 | Remaining: 53 | Reset: 51s
   ✅ 31 ta yozuv olindi
[8/2256] Sensor 9648796 | 2025-08 | Remaining: 52 | Reset: 50s
   ✅ 31 ta yozuv olindi
[9/2256] Sensor 9648796 | 2025-09 | Remaining: 51 | Reset: 48s
   ✅ 30 ta yozuv olindi
[10/2256] Sensor 9648796 | 2025-10 | Remaining: 50 | Reset: 46s
   ✅ 31 ta yozuv olindi
[11/2256] Sensor 9648796 | 2025-11 | Remaining: 49 | Reset: 45s
   ✅ 30 ta yozuv olindi
[12/2256] Sensor 9648796 | 2025-12 | Rema

In [ ]:
# Cell 3: Ma'lumotlarni tekislash (flatten) va DataFrame yaratish
from pyspark.sql.functions import col, lit

if all_measurements:
    flat_records = []
    
    for rec in all_measurements:
        flat = {
            "sensor_id":        rec.get("sensor_id"),
            "value":            rec.get("value"),
            "parameter":        rec.get("parameter", {}).get("name") if rec.get("parameter") else None,
            "unit":             rec.get("parameter", {}).get("units") if rec.get("parameter") else None,
            "date_from":        rec.get("period", {}).get("datetimeFrom", {}).get("utc") if rec.get("period") else None,
            "date_to":          rec.get("period", {}).get("datetimeTo", {}).get("utc") if rec.get("period") else None,
            "coverage":         rec.get("coverage"),
            "summary_min":      rec.get("summary", {}).get("min") if rec.get("summary") else None,
            "summary_max":      rec.get("summary", {}).get("max") if rec.get("summary") else None,
            "summary_avg":      rec.get("summary", {}).get("mean") if rec.get("summary") else None,
            "summary_sd":       rec.get("summary", {}).get("sd") if rec.get("summary") else None,
        }
        flat_records.append(flat)

    # Pandas orqali Spark DataFrame ga o'tkazish
    import pandas as pd
    pandas_df = pd.DataFrame(flat_records)
    daily_df = spark.createDataFrame(pandas_df)

    print("📋 Schema:")
    daily_df.printSchema()
    print(f"\n📊 Jami qatorlar: {daily_df.count()}")
    daily_df.show(10, truncate=False)

else:
    print("⚠️ Hech qanday ma'lumot olinmadi")

In [ ]:
# Cell 3.5 - Schemani tekislash
from pyspark.sql.functions import col

daily_df_clean = daily_df.select(
    col("sensor_id"),
    col("value"),
    col("parameter"),
    col("unit"),
    col("date_from"),
    col("date_to"),
    
    # coverage ichidan kerakli ustunlar
    col("coverage.datetimeFrom.utc").alias("coverage_from"),
    col("coverage.datetimeTo.utc").alias("coverage_to"),
    col("coverage.expectedCount").alias("expected_count"),
    col("coverage.observedCount").alias("observed_count"),
    col("coverage.percentComplete").alias("percent_complete"),
    col("coverage.percentCoverage").alias("percent_coverage"),
    
    col("summary_min"),
    col("summary_max"),
    col("summary_sd"),
)

print("✅ Yangi schema:")
daily_df_clean.printSchema()
daily_df_clean.show(5, truncate=False)

In [ ]:
# Cell 5 - Tekislash va Delta ga saqlash
from pyspark.sql.functions import col, to_timestamp, date_format, hour

daily_df_clean = daily_df.select(
    col("sensor_id"),
    col("value"),
    col("parameter"),
    col("unit"),

    to_timestamp(col("date_from")).alias("datetime_from"),
    date_format(to_timestamp(col("date_from")), "yyyy-MM-dd").alias("sana_from"),
    hour(to_timestamp(col("date_from"))).alias("soat_from"),

    to_timestamp(col("date_to")).alias("datetime_to"),
    date_format(to_timestamp(col("date_to")), "yyyy-MM-dd").alias("sana_to"),
    hour(to_timestamp(col("date_to"))).alias("soat_to"),

    col("coverage.expectedCount").alias("expected_count"),
    col("coverage.observedCount").alias("observed_count"),
    col("coverage.percentComplete").alias("percent_complete"),
    col("coverage.percentCoverage").alias("percent_coverage"),

    col("summary_min"),
    col("summary_max"),
    col("summary_sd"),
)

# # Delta ga saqlash
# daily_df_clean.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable("silver_openaq_daily_2025")

print("✅ Saqlandi!")
print(f"Jami qatorlar: {daily_df_clean.count()}")
display(daily_df_clean)